In [ ]:
!pip install rdkit

!pip install torch-geometric
!pip install tensorflow

In [ ]:
# If needed in Colab, uncomment:
!pip install rdkit-pypi torch-geometric
# RDKit installation (for Colab)
#!pip install rdkit-pypi
!pip install rdkit
# Runtime restart gerekebilir
#import os
#os.kill(os.getpid(), 9)

In [ ]:
!pip install torch-geometric

In [2]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import rdmolops
from rdkit.Chem import HybridizationType

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve


In [ ]:
SEED = 42
BATCH_SIZE = 64
EPOCHS = 50
PATIENCE = 10
LR = 3e-4
WEIGHT_DECAY = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print("Using device:", DEVICE)


In [ ]:
# Load data
# Replace path if needed
CSV_PATH = "https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv"
df = pd.read_csv(CSV_PATH)

# keep valid rows
df = df[["smiles", "HIV_active"]].dropna().reset_index(drop=True)
df["HIV_active"] = df["HIV_active"].astype(int)
print(df.shape)
df.head()


In [ ]:
def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetTotalDegree(),
        atom.GetFormalCharge(),
        int(atom.GetHybridization() == HybridizationType.SP),
        int(atom.GetHybridization() == HybridizationType.SP2),
        int(atom.GetHybridization() == HybridizationType.SP3),
        int(atom.GetIsAromatic()),
        int(atom.IsInRing())
    ]

def mol_to_graph(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    adj = rdmolops.GetAdjacencyMatrix(mol)
    if adj.shape[0] == 0:
        return None

    x = torch.tensor([atom_features(atom) for atom in mol.GetAtoms()], dtype=torch.float)
    edge_index = torch.tensor(np.array(np.nonzero(adj)), dtype=torch.long)
    y = torch.tensor([label], dtype=torch.float)

    return Data(x=x, edge_index=edge_index, y=y)

graphs = []
for _, row in df.iterrows():
    g = mol_to_graph(row["smiles"], row["HIV_active"])
    if g is not None:
        graphs.append(g)

valid_idx = [i for i, row in enumerate(df.itertuples(index=False)) if mol_to_graph(row.smiles, row.HIV_active) is not None]
df = df.iloc[valid_idx].reset_index(drop=True)

print("Valid graphs:", len(graphs))


In [6]:
class GCNNet(nn.Module):
    def __init__(self, in_channels=8, dropout=0.35):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.conv2 = GCNConv(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.conv3 = GCNConv(64, 32)
        self.dropout = nn.Dropout(dropout)
        self.lin1 = nn.Linear(32, 16)
        self.lin2 = nn.Linear(16, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = self.dropout(x)
        x = F.relu(self.lin1(x))
        x = self.dropout(x)
        x = torch.sigmoid(self.lin2(x)).view(-1)
        return x


In [7]:
def make_loader(indexes, batch_size=BATCH_SIZE, shuffle=False):
    subset = [graphs[i] for i in indexes]
    return DataLoader(subset, batch_size=batch_size, shuffle=shuffle)

def evaluate(loader, model, loss_fn):
    model.eval()
    total_loss = 0.0
    probs_all, y_all = [], []

    with torch.no_grad():
        for data in loader:
            data = data.to(DEVICE)
            probs = model(data)
            y = data.y.view(-1).float()
            loss = loss_fn(probs, y)
            total_loss += loss.item() * y.size(0)

            probs_all.extend(probs.detach().cpu().numpy())
            y_all.extend(y.detach().cpu().numpy())

    probs_all = np.array(probs_all)
    y_all = np.array(y_all).astype(int)

    # threshold-independent metrics
    auc = roc_auc_score(y_all, probs_all) if len(np.unique(y_all)) > 1 else np.nan
    avg_loss = total_loss / max(len(y_all), 1)
    return avg_loss, y_all, probs_all, auc

def find_best_threshold(y_true, probs):
    thresholds = np.arange(0.10, 0.91, 0.02)
    best_thr, best_f1 = 0.50, -1
    for thr in thresholds:
        pred = (probs >= thr).astype(int)
        score = f1_score(y_true, pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thr = thr
    return float(best_thr)

def train_one_fold(train_idx, val_idx, test_idx, fold_id):
    set_seed(SEED + fold_id)

    train_loader = make_loader(train_idx, shuffle=True)
    val_loader = make_loader(val_idx, shuffle=False)
    test_loader = make_loader(test_idx, shuffle=False)

    model = GCNNet().to(DEVICE)

    y_train = df.loc[train_idx, "HIV_active"].values
    pos = np.sum(y_train == 1)
    neg = np.sum(y_train == 0)
    pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float, device=DEVICE)

    class WeightedBCELoss(nn.Module):
        def __init__(self, pos_weight):
            super().__init__()
            self.pos_weight = pos_weight
        def forward(self, probs, target):
            eps = 1e-7
            probs = torch.clamp(probs, eps, 1 - eps)
            loss = -(self.pos_weight * target * torch.log(probs) + (1 - target) * torch.log(1 - probs))
            return loss.mean()

    loss_fn = WeightedBCELoss(pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "train_auc": [], "val_auc": []}
    best_state = None
    best_val_auc = -np.inf
    wait = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        train_probs, train_y = [], []

        for data in train_loader:
            data = data.to(DEVICE)
            optimizer.zero_grad()
            probs = model(data)
            y = data.y.view(-1).float()
            loss = loss_fn(probs, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * y.size(0)
            train_probs.extend(probs.detach().cpu().numpy())
            train_y.extend(y.detach().cpu().numpy())

        train_probs = np.array(train_probs)
        train_y = np.array(train_y).astype(int)
        train_thr = find_best_threshold(train_y, train_probs)
        train_pred = (train_probs >= train_thr).astype(int)
        train_loss = total_loss / max(len(train_y), 1)
        train_acc = accuracy_score(train_y, train_pred)
        train_auc = roc_auc_score(train_y, train_probs) if len(np.unique(train_y)) > 1 else np.nan

        val_loss, val_y, val_probs, val_auc = evaluate(val_loader, model, loss_fn)
        val_thr = find_best_threshold(val_y, val_probs)
        val_pred = (val_probs >= val_thr).astype(int)
        val_acc = accuracy_score(val_y, val_pred)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_auc"].append(train_auc)
        history["val_auc"].append(val_auc)

        scheduler.step(val_auc)

        if val_auc > best_val_auc + 1e-4:
            best_val_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    model.load_state_dict(best_state)

    # pick threshold from best validation predictions
    _, val_y, val_probs, _ = evaluate(val_loader, model, loss_fn)
    best_threshold = find_best_threshold(val_y, val_probs)

    _, test_y, test_probs, test_auc = evaluate(test_loader, model, loss_fn)
    test_pred = (test_probs >= best_threshold).astype(int)

    fold_result = {
        "fold": fold_id,
        "test_accuracy": accuracy_score(test_y, test_pred),
        "test_precision": precision_score(test_y, test_pred, zero_division=0),
        "test_recall": recall_score(test_y, test_pred, zero_division=0),
        "test_f1": f1_score(test_y, test_pred, zero_division=0),
        "test_roc_auc": test_auc,
        "best_threshold": best_threshold,
        "epochs_ran": len(history["train_loss"])
    }
    return fold_result, history


In [ ]:
all_fold_results = []
all_histories = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
indices = np.arange(len(df))
y = df["HIV_active"].values

for fold_id, (train_idx, temp_idx) in enumerate(skf.split(indices, y), start=1):
    temp_y = y[temp_idx]
    val_rel_idx, test_rel_idx = train_test_split(
        np.arange(len(temp_idx)),
        test_size=0.5,
        random_state=SEED + fold_id,
        stratify=temp_y
    )
    val_idx = temp_idx[val_rel_idx]
    test_idx = temp_idx[test_rel_idx]

    print(f"\n===== Fold {fold_id} =====")
    print("Train:", len(train_idx), "Val:", len(val_idx), "Test:", len(test_idx))

    fold_result, hist = train_one_fold(train_idx, val_idx, test_idx, fold_id)
    print(fold_result)

    all_fold_results.append(fold_result)
    all_histories.append(hist)

results_df = pd.DataFrame(all_fold_results)
results_df


In [ ]:
summary_rows = []
metric_cols = ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]

for col in metric_cols:
    mean = results_df[col].mean()
    std = results_df[col].std(ddof=1)
    var = results_df[col].var(ddof=1)
    summary_rows.append({
        "Metric": col.replace("test_", "").upper(),
        "Mean": mean,
        "Std": std,
        "Variance": var,
        "Formatted": f"{mean:.3f} ± {std:.3f}"
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:
final_table = pd.DataFrame([{
    "Model": "GCN",
    "Accuracy": summary_df.loc[summary_df["Metric"]=="ACCURACY", "Formatted"].iloc[0],
    "Precision": summary_df.loc[summary_df["Metric"]=="PRECISION", "Formatted"].iloc[0],
    "Recall": summary_df.loc[summary_df["Metric"]=="RECALL", "Formatted"].iloc[0],
    "F1": summary_df.loc[summary_df["Metric"]=="F1", "Formatted"].iloc[0],
    "ROC-AUC": summary_df.loc[summary_df["Metric"]=="ROC_AUC", "Formatted"].iloc[0],
}])
final_table


In [ ]:
results_df.to_csv("GCN_fold_results.csv", index=False)
summary_df.to_csv("GCN_summary_results.csv", index=False)
final_table.to_csv("GCN_final_table.csv", index=False)
print("Saved: GCN_fold_results.csv, GCN_summary_results.csv, GCN_final_table.csv")


In [ ]:
# Average training curves across folds
max_len = max(len(h["train_loss"]) for h in all_histories)

def pad_series(series, target_len):
    if len(series) == target_len:
        return np.array(series, dtype=float)
    arr = np.array(series, dtype=float)
    pad_value = arr[-1]
    return np.pad(arr, (0, target_len - len(arr)), constant_values=pad_value)

avg_train_acc = np.mean([pad_series(h["train_acc"], max_len) for h in all_histories], axis=0)
avg_val_acc   = np.mean([pad_series(h["val_acc"], max_len) for h in all_histories], axis=0)
avg_train_loss = np.mean([pad_series(h["train_loss"], max_len) for h in all_histories], axis=0)
avg_val_loss   = np.mean([pad_series(h["val_loss"], max_len) for h in all_histories], axis=0)

epochs = np.arange(1, max_len + 1)

plt.figure(figsize=(14,5))
plt.subplot(1,2,1)
plt.plot(epochs, avg_train_acc, label="Train")
plt.plot(epochs, avg_val_acc, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("GCN Average Classification Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs, avg_train_loss, label="Train")
plt.plot(epochs, avg_val_loss, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GCN Average Loss")
plt.legend()

plt.tight_layout()
plt.show()


## GCN docking preparation block

This block is prepared for Colab.

Added workflow:
- RDKit installation cell
- select the best fold
- rebuild the same fold
- retrain the model
- top 10 candidates
- shared 13 columns
- final 2 candidates
- colored 2D molecule drawing
- `.smi` docking file

In [ ]:
# Colab RDKit install
import sys, subprocess, pkgutil

if pkgutil.find_loader("rdkit") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit-pypi"])

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import Lipinski as RdLipinski, Crippen, QED

print("RDKit OK")

In [ ]:
# ==========================================
# GCN FINAL PIPELINE (SELF-CONTAINED, ROBUST)
# ==========================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch_geometric.loader import DataLoader

# 1) Select the best fold
auc_col = "test_roc_auc" if "test_roc_auc" in results_df.columns else results_df.columns[-1]
best_fold_idx = int(results_df[auc_col].astype(float).idxmax())
best_fold_number = best_fold_idx + 1

print(f"Using AUC column: {auc_col}")
print(f"Using best fold: {best_fold_number}")

# 2) Get labels from df
if "HIV_active" not in df.columns:
    raise ValueError("Column 'HIV_active' not found in df.")

labels = df["HIV_active"].astype(int).values

# 3) Rebuild the same split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
indices = np.arange(len(graphs))
splits = list(skf.split(indices, labels))

train_idx, temp_idx = splits[best_fold_idx]
train_idx = np.array(train_idx)
temp_idx = np.array(temp_idx)

temp_labels = np.array(labels)[temp_idx]
val_sub_idx, test_sub_idx = train_test_split(
    np.arange(len(temp_idx)),
    test_size=0.5,
    random_state=SEED + best_fold_number,
    stratify=temp_labels
)

train_graphs = [graphs[i] for i in train_idx]
val_graphs   = [graphs[temp_idx[i]] for i in val_sub_idx]
test_graphs  = [graphs[temp_idx[i]] for i in test_sub_idx]

test_global_idx = temp_idx[test_sub_idx]
smiles_test = df.iloc[test_global_idx]["smiles"].reset_index(drop=True)
y_test = df.iloc[test_global_idx]["HIV_active"].astype(float).values

print("Train:", len(train_graphs), "Val:", len(val_graphs), "Test:", len(test_graphs))

# 3) DataLoader
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

# 4) Rebuild and train the model
set_seed(SEED + best_fold_number)

model = GCNModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.BCELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

best_val_loss = float("inf")
best_state_dict = None
wait = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = loss_fn(out, data.y.view(-1).float())
        loss.backward()
        optimizer.step()

    model.eval()
    val_total_loss = 0.0
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            out = model(data)
            loss = loss_fn(out, data.y.view(-1).float())
            val_total_loss += loss.item() * data.num_graphs

    val_loss = val_total_loss / len(val_loader.dataset)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state_dict)
model = model.to(device)
model.eval()

# 5) Test prediction
test_preds = []
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out = model(data)
        test_preds.extend(out.detach().cpu().numpy().tolist())

y_prob = np.array(test_preds).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

# 6) Prediction dataframe
df_pred = pd.DataFrame({
    "fold": best_fold_number,
    "smiles": smiles_test,
    "y_true": y_test,
    "y_pred": y_pred,
    "y_prob": y_prob
})

# 7) Top 10 candidates
top_df = df_pred.sort_values(by="y_prob", ascending=False).head(10).reset_index(drop=True)

# 8) Descriptor hesaplama
def compute_desc(sm):
    mol = Chem.MolFromSmiles(sm)
    if mol is None:
        return None

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = RdLipinski.NumHDonors(mol)
    hba = RdLipinski.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rot = RdLipinski.NumRotatableBonds(mol)
    qed = QED.qed(mol)
    lip = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)

    return {
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "TPSA": tpsa,
        "RotatableBonds": rot,
        "QED": qed,
        "Lipinski": lip
    }

rows = []
for _, row in top_df.iterrows():
    d = compute_desc(row["smiles"])
    if d is not None:
        d.update(row.to_dict())
        rows.append(d)

df_desc = pd.DataFrame(rows)

# 9) Shared 13 columns
df_desc = df_desc[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nGCN TOP 10 CANDIDATES:")
display(df_desc)

# 10) Final 2 candidates
filtered_df = df_desc[df_desc["Lipinski"] == True].copy()

if len(filtered_df) >= 2:
    final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()
else:
    final_df = df_desc.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()

final_df = final_df[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nGCN FINAL 2 CANDIDATES:")
display(final_df)

# 11) Save
df_desc.to_csv("GCN_top_10_candidates.csv", index=False)
final_df.to_csv("GCN_final_2_candidates.csv", index=False)
final_df["smiles"].to_csv("GCN_docking_input.smi", index=False, header=False)

print("\nSaved: GCN_top_10_candidates.csv")
print("Saved: GCN_final_2_candidates.csv")
print("Saved: GCN_docking_input.smi")

# 12) Colored drawing
mols = [Chem.MolFromSmiles(sm) for sm in final_df["smiles"]]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=2,
    subImgSize=(340, 340),
    legends=[
        f"GCN Mol1\nProb={final_df.iloc[0]['y_prob']:.3f}" if len(final_df) > 0 else "",
        f"GCN Mol2\nProb={final_df.iloc[1]['y_prob']:.3f}" if len(final_df) > 1 else ""
    ]
)

display(img)

In [ ]:
# ==========================================
# GCN FINAL PIPELINE (SELF-CONTAINED, ROBUST)
# ==========================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch_geometric.loader import DataLoader

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import Lipinski as RdLipinski, Crippen, QED

# 1) Select the best fold
auc_col = "test_roc_auc" if "test_roc_auc" in results_df.columns else results_df.columns[-1]
best_fold_idx = int(results_df[auc_col].astype(float).idxmax())
best_fold_number = best_fold_idx + 1

print(f"Using AUC column: {auc_col}")
print(f"Using best fold: {best_fold_number}")

# 2) define labels
if "HIV_active" not in df.columns:
    raise ValueError("Column 'HIV_active' not found in df.")

labels = df["HIV_active"].astype(int).values

# 3) Rebuild the same split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
indices = np.arange(len(graphs))
splits = list(skf.split(indices, labels))

train_idx, temp_idx = splits[best_fold_idx]
train_idx = np.array(train_idx)
temp_idx = np.array(temp_idx)

temp_labels = labels[temp_idx]
val_sub_idx, test_sub_idx = train_test_split(
    np.arange(len(temp_idx)),
    test_size=0.5,
    random_state=SEED + best_fold_number,
    stratify=temp_labels
)

train_graphs = [graphs[i] for i in train_idx]
val_graphs   = [graphs[temp_idx[i]] for i in val_sub_idx]
test_graphs  = [graphs[temp_idx[i]] for i in test_sub_idx]

test_global_idx = temp_idx[test_sub_idx]
smiles_test = df.iloc[test_global_idx]["smiles"].reset_index(drop=True)
y_test = df.iloc[test_global_idx]["HIV_active"].astype(float).values

print("Train:", len(train_graphs), "Val:", len(val_graphs), "Test:", len(test_graphs))

# 4) DataLoader
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

# 5) Rebuild and train the model
set_seed(SEED + best_fold_number)

model = GCNModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.BCELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

best_val_loss = float("inf")
best_state_dict = None
wait = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = loss_fn(out, data.y.view(-1).float())
        loss.backward()
        optimizer.step()

    model.eval()
    val_total_loss = 0.0
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            out = model(data)
            loss = loss_fn(out, data.y.view(-1).float())
            val_total_loss += loss.item() * data.num_graphs

    val_loss = val_total_loss / len(val_loader.dataset)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state_dict)
model = model.to(device)
model.eval()

# 6) Test prediction
test_preds = []
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out = model(data)
        test_preds.extend(out.detach().cpu().numpy().tolist())

y_prob = np.array(test_preds).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

# 7) Prediction dataframe
df_pred = pd.DataFrame({
    "fold": best_fold_number,
    "smiles": smiles_test,
    "y_true": y_test,
    "y_pred": y_pred,
    "y_prob": y_prob
})

# 8) Top 10 candidates
top_df = df_pred.sort_values(by="y_prob", ascending=False).head(10).reset_index(drop=True)

# 9) Descriptor hesaplama
def compute_desc(sm):
    mol = Chem.MolFromSmiles(sm)
    if mol is None:
        return None

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = RdLipinski.NumHDonors(mol)
    hba = RdLipinski.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rot = RdLipinski.NumRotatableBonds(mol)
    qed = QED.qed(mol)
    lip = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)

    return {
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "TPSA": tpsa,
        "RotatableBonds": rot,
        "QED": qed,
        "Lipinski": lip
    }

rows = []
for _, row in top_df.iterrows():
    d = compute_desc(row["smiles"])
    if d is not None:
        d.update(row.to_dict())
        rows.append(d)

df_desc = pd.DataFrame(rows)

# 10) Shared 13 columns
df_desc = df_desc[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nGCN TOP 10 CANDIDATES:")
display(df_desc)

# 11) Final 2 candidates
filtered_df = df_desc[df_desc["Lipinski"] == True].copy()

if len(filtered_df) >= 2:
    final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()
else:
    final_df = df_desc.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()

final_df = final_df[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nGCN FINAL 2 CANDIDATES:")
display(final_df)

# 12) Save
df_desc.to_csv("GCN_top_10_candidates.csv", index=False)
final_df.to_csv("GCN_final_2_candidates.csv", index=False)
final_df["smiles"].to_csv("GCN_docking_input.smi", index=False, header=False)

print("\nSaved: GCN_top_10_candidates.csv")
print("Saved: GCN_final_2_candidates.csv")
print("Saved: GCN_docking_input.smi")

# 13) Colored drawing
mols = [Chem.MolFromSmiles(sm) for sm in final_df["smiles"]]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=2,
    subImgSize=(340, 340),
    legends=[
        f"GCN Mol1\nProb={final_df.iloc[0]['y_prob']:.3f}" if len(final_df) > 0 else "",
        f"GCN Mol2\nProb={final_df.iloc[1]['y_prob']:.3f}" if len(final_df) > 1 else ""
    ]
)

display(img)

In [ ]:
# ==========================================
# GCN FINAL PIPELINE (FULLY ROBUST)
# ==========================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch_geometric.loader import DataLoader

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import Lipinski as RdLipinski, Crippen, QED

# 0) define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 1) Select the best fold
auc_col = "test_roc_auc" if "test_roc_auc" in results_df.columns else results_df.columns[-1]
best_fold_idx = int(results_df[auc_col].astype(float).idxmax())
best_fold_number = best_fold_idx + 1

print(f"Using AUC column: {auc_col}")
print(f"Using best fold: {best_fold_number}")

# 2) define labels
if "HIV_active" not in df.columns:
    raise ValueError("Column 'HIV_active' not found in df.")

labels = df["HIV_active"].astype(int).values

# 3) Rebuild the same split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
indices = np.arange(len(graphs))
splits = list(skf.split(indices, labels))

train_idx, temp_idx = splits[best_fold_idx]
train_idx = np.array(train_idx)
temp_idx = np.array(temp_idx)

temp_labels = labels[temp_idx]
val_sub_idx, test_sub_idx = train_test_split(
    np.arange(len(temp_idx)),
    test_size=0.5,
    random_state=SEED + best_fold_number,
    stratify=temp_labels
)

train_graphs = [graphs[i] for i in train_idx]
val_graphs   = [graphs[temp_idx[i]] for i in val_sub_idx]
test_graphs  = [graphs[temp_idx[i]] for i in test_sub_idx]

test_global_idx = temp_idx[test_sub_idx]
smiles_test = df.iloc[test_global_idx]["smiles"].reset_index(drop=True)
y_test = df.iloc[test_global_idx]["HIV_active"].astype(float).values

print("Train:", len(train_graphs), "Val:", len(val_graphs), "Test:", len(test_graphs))

# 4) DataLoader
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

# 5) Automatically find the model class
model_class = None
candidate_names = [
    "GCNModel", "GCNNet", "GCN", "GraphGCN", "Net", "Model"
]

for name in candidate_names:
    if name in globals() and isinstance(globals()[name], type):
        model_class = globals()[name]
        print(f"Using model class: {name}")
        break

if model_class is None:
    for name, obj in list(globals().items()):
        if isinstance(obj, type) and "gcn" in name.lower():
            model_class = obj
            print(f"Using detected model class: {name}")
            break

if model_class is None:
    raise ValueError("GCN model class not found.")

# 6) Fallback if seed function is missing
if "set_seed" in globals():
    set_seed(SEED + best_fold_number)
else:
    torch.manual_seed(SEED + best_fold_number)
    np.random.seed(SEED + best_fold_number)

# 7) Modeli kur
try:
    model = model_class().to(device)
except TypeError:
    try:
        model = model_class(num_node_features=train_graphs[0].x.shape[1]).to(device)
    except TypeError:
        try:
            model = model_class(input_dim=train_graphs[0].x.shape[1]).to(device)
        except TypeError:
            try:
                model = model_class(in_channels=train_graphs[0].x.shape[1]).to(device)
            except TypeError:
                raise ValueError("Model class was found but could not be initialized with suitable parameters.")

# 8) Hiperparametre fallback
lr_value = LR if "LR" in globals() else 1e-3
wd_value = WEIGHT_DECAY if "WEIGHT_DECAY" in globals() else 1e-5
epochs_value = NUM_EPOCHS if "NUM_EPOCHS" in globals() else 50
patience_value = PATIENCE if "PATIENCE" in globals() else 10

optimizer = torch.optim.Adam(model.parameters(), lr=lr_value, weight_decay=wd_value)
loss_fn = nn.BCELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

best_val_loss = float("inf")
best_state_dict = None
wait = 0

# 9) Training
for epoch in range(1, epochs_value + 1):
    model.train()
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        out = out.view(-1)
        loss = loss_fn(out, data.y.view(-1).float())
        loss.backward()
        optimizer.step()

    model.eval()
    val_total_loss = 0.0
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            out = model(data)
            out = out.view(-1)
            loss = loss_fn(out, data.y.view(-1).float())
            val_total_loss += loss.item() * data.num_graphs

    val_loss = val_total_loss / len(val_loader.dataset)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience_value:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state_dict)
model = model.to(device)
model.eval()

# 10) Test prediction
test_preds = []
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out = model(data)
        out = out.view(-1)
        test_preds.extend(out.detach().cpu().numpy().tolist())

y_prob = np.array(test_preds).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

# 11) Prediction dataframe
df_pred = pd.DataFrame({
    "fold": best_fold_number,
    "smiles": smiles_test,
    "y_true": y_test,
    "y_pred": y_pred,
    "y_prob": y_prob
})

# 12) Top 10 candidates
top_df = df_pred.sort_values(by="y_prob", ascending=False).head(10).reset_index(drop=True)

# 13) Descriptor hesaplama
def compute_desc(sm):
    mol = Chem.MolFromSmiles(sm)
    if mol is None:
        return None

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = RdLipinski.NumHDonors(mol)
    hba = RdLipinski.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rot = RdLipinski.NumRotatableBonds(mol)
    qed = QED.qed(mol)
    lip = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)

    return {
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "TPSA": tpsa,
        "RotatableBonds": rot,
        "QED": qed,
        "Lipinski": lip
    }

rows = []
for _, row in top_df.iterrows():
    d = compute_desc(row["smiles"])
    if d is not None:
        d.update(row.to_dict())
        rows.append(d)

df_desc = pd.DataFrame(rows)

# 14) Shared 13 columns
df_desc = df_desc[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nGCN TOP 10 CANDIDATES:")
display(df_desc)

# 15) Final 2 candidates
filtered_df = df_desc[df_desc["Lipinski"] == True].copy()

if len(filtered_df) >= 2:
    final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()
else:
    final_df = df_desc.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()

final_df = final_df[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotatableBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nGCN FINAL 2 CANDIDATES:")
display(final_df)

# 16) Save
df_desc.to_csv("GCN_top_10_candidates.csv", index=False)
final_df.to_csv("GCN_final_2_candidates.csv", index=False)
final_df["smiles"].to_csv("GCN_docking_input.smi", index=False, header=False)

print("\nSaved: GCN_top_10_candidates.csv")
print("Saved: GCN_final_2_candidates.csv")
print("Saved: GCN_docking_input.smi")

# 17) Colored drawing
mols = [Chem.MolFromSmiles(sm) for sm in final_df["smiles"]]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=2,
    subImgSize=(340, 340),
    legends=[
        f"GCN Mol1\nProb={final_df.iloc[0]['y_prob']:.3f}" if len(final_df) > 0 else "",
        f"GCN Mol2\nProb={final_df.iloc[1]['y_prob']:.3f}" if len(final_df) > 1 else ""
    ]
)

display(img)